In [1]:
!nvidia-smi

Wed Mar 25 08:38:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install transformers datasets peft accelerate bitsandbytes trl -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 43.7 MB/s eta 0:00:00


In [3]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from datasets import load_dataset

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [4]:
# Load famous Dolly dataset — 15k Q&A pairs
dataset = load_dataset("databricks/databricks-dolly-15k", split="train")

print(f"Total samples: {len(dataset)}")
print(f"\nSample entry:")
print(f"Category: {dataset[0]['category']}")
print(f"Instruction: {dataset[0]['instruction']}")
print(f"Response: {dataset[0]['response'][:200]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Total samples: 15011

Sample entry:
Category: closed_qa
Instruction: When did Virgin Australia start operating?
Response: Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.


In [5]:
def format_prompt(sample):
    """
    Convert each sample into instruction format.
    This is the exact format TinyLlama was trained on.
    """
    instruction = sample["instruction"]
    context     = sample.get("context", "")
    response    = sample["response"]

    if context:
        text = f"""<|system|>
You are a helpful assistant.</s>
<|user|>
{instruction}

Context: {context}</s>
<|assistant|>
{response}</s>"""
    else:
        text = f"""<|system|>
You are a helpful assistant.</s>
<|user|>
{instruction}</s>
<|assistant|>
{response}</s>"""

    return {"text": text}

# Apply formatting to entire dataset
dataset = dataset.map(format_prompt)

# Use only 1000 samples for fast training on free Colab
dataset = dataset.select(range(500))

print(f"Training samples: {len(dataset)}")
print(f"\nFormatted sample:")
print(dataset[0]["text"][:400])

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

Training samples: 500

Formatted sample:
<|system|>
You are a helpful assistant.</s>
<|user|>
When did Virgin Australia start operating?

Context: Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a m


In [6]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# 4-bit quantization config — this is what makes QLoRA possible
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # load model in 4-bit
    bnb_4bit_quant_type="nf4",              # NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.float16,   # compute in float16
    bnb_4bit_use_double_quant=True,         # double quantization for extra savings
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Load model in 4-bit
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

✓ Model loaded: TinyLlama/TinyLlama-1.1B-Chat-v1.0
  Parameters: 615,606,272


In [7]:
def generate_response(prompt, max_new_tokens=200):
    """Generate a response from the model."""
    formatted = f"<|user|>\n{prompt}</s>\n<|assistant|>\n"
    inputs    = tokenizer(formatted, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("<|assistant|>")[-1].strip()

# Test before fine-tuning
print("=" * 50)
print("BEFORE FINE-TUNING:")
print("=" * 50)
print(generate_response("What is machine learning?"))

BEFORE FINE-TUNING:
Machine learning (ML) is a type of artificial intelligence (AI) that enables computers to learn and improve from experience without being explicitly programmed. It's a form of AI that enables computers to make predictions, identify patterns, and solve complex problems based on data. The most popular ML algorithms include:

1. Decision Trees: A decision tree is a type of algorithm that analyzes data and assigns a label (positive or negative) to each observation. It is a widely used tool in ML for classification and regression tasks.

2. Random Forests: A random forest is a type of ensemble learning algorithm that uses a variety of decision trees to improve accuracy. It is commonly used in classification and regression tasks.

3. Gradient Boosting: Gradient boosting is a type of boosted trees algorithm that uses multiple decision trees to improve accuracy. It is commonly used in regression and classification tasks.

4. Support Vector Machines


In [8]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,                          # rank — size of LoRA matrices
    lora_alpha=32,                 # scaling factor (usually 2x rank)
    target_modules=[               # which layers to apply LoRA to
        "q_proj",
        "v_proj",
        "k_proj",
        "o_proj",
    ],
    lora_dropout=0.05,             # dropout for regularization
    bias="none",
    task_type=TaskType.CAUSAL_LM,  # causal language modeling
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)

# See how many parameters we're actually training
model.print_trainable_parameters()
# Output will be something like:
# trainable params: 2,097,152 || all params: 1,102,048,256 || trainable%: 0.19
# Only 0.19% of parameters are being trained!

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [9]:
training_args = TrainingArguments(
    output_dir="./tinyllama-finetuned",
    num_train_epochs=2,              # 2 passes through dataset
    per_device_train_batch_size=4,   # 4 samples per step
    gradient_accumulation_steps=4,   # accumulate gradients for 4 steps
    learning_rate=2e-4,              # learning rate
    fp16=True,                       # use float16 for speed
    logging_steps=50,                # print loss every 50 steps
    save_steps=200,                  # save checkpoint every 200 steps
    warmup_ratio=0.03,               # warmup for 3% of training
    lr_scheduler_type="cosine",      # cosine learning rate decay
    report_to="none",                # don't use wandb
)

print("✓ Training config ready")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✓ Training config ready
  Epochs: 2
  Batch size: 4
  Learning rate: 0.0002


In [10]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=TrainingArguments(
        output_dir="./tinyllama-finetuned",
        num_train_epochs=2,
        per_device_train_batch_size=1,      # ← reduced from 4 to 1
        gradient_accumulation_steps=8,      # ← increased to compensate
        learning_rate=2e-4,
        bf16=True,
        logging_steps=25,
        save_steps=200,
        warmup_steps=10,
        lr_scheduler_type="cosine",
        report_to="none",
    ),
)

print("Starting training...")
trainer.train()
print("\n✓ Training complete!")

Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2280 > 2048). Running this sequence through the model will result in indexing errors


Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting training...


Step,Training Loss
25,1.701387
50,1.588766
75,1.519250
100,1.461511
125,1.536966



✓ Training complete!


In [11]:
print("=" * 50)
print("AFTER FINE-TUNING:")
print("=" * 50)
print(generate_response("What is machine learning?"))

print("\n" + "=" * 50)
print("More tests:")
print("=" * 50)
print("\nQ: What is supervised learning?")
print(generate_response("What is supervised learning?"))

print("\nQ: Explain neural networks simply")
print(generate_response("Explain neural networks simply"))

AFTER FINE-TUNING:
Machine learning is a field of artificial intelligence that applies statistical models to large amounts of data. The goal of machine learning is to improve the accuracy of predictions based on the data. Machine learning algorithms can be classified into supervised and unsupervised learning. Supervised learning is the most common, where the goal is to predict an output based on a set of input data. For example, the goal is to predict the outcome of an auction given the number of bids, price, and auction history. Unsupervised learning is the goal to find new patterns in data, which may not have necessarily a predictive value. For example, the goal is to find clusters of similar data.

Some common types of machine learning algorithms are Support Vector Machines (SVM), Gradient Boosting, Random Forest, Neural Networks, and Deep Learning.

More tests:

Q: What is supervised learning?
Supervised learning is the process of learning a model from a labeled dataset. The model 

In [12]:
model.save_pretrained("./tinyllama-lora-weights")
print("✓ Model saved!")

✓ Model saved!


In [13]:
from huggingface_hub import login

login()

In [14]:
# Push LoRA weights to HuggingFace Hub
model.push_to_hub("tinyllama-dolly-finetuned")
tokenizer.push_to_hub("tinyllama-dolly-finetuned")

print("✓ Model pushed to HuggingFace Hub!")
print("  Visit: https://huggingface.co/Mehak-123-arora/tinyllama-dolly-finetuned")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   7%|6         |  623kB / 9.03MB            

No files have been modified since last commit. Skipping to prevent empty commit.


✓ Model pushed to HuggingFace Hub!
  Visit: https://huggingface.co/Mehak-123-arora/tinyllama-dolly-finetuned
